# MedNorm-VI S1 - Mention First-Run Smoke Readiness

**SMOKE_ONLY. Colab Pro GPU runtime required.**

## TWO-PASS EXECUTION (required)

This notebook installs dependencies and then **forces one kernel restart**, because
changing packages inside a live kernel is what corrupted the NumPy C-ABI previously
(`numpy.dtype size changed ... Expected 96 from C header, got 88 from PyObject`).

1. **PASS 1** - `Runtime > Run all`. The notebook installs the constrained dependency
   set, writes a bootstrap marker, and **restarts the kernel on purpose**.
2. **PASS 2** - after the restart, choose `Runtime > Run all` **again**. The marker is
   found, installation is skipped, the ABI preflight runs, and the smoke proceeds.

The marker is **not** trusted on a version string alone. It records the dependency
contract version, the SHA-256 of the contract file bytes, the Python `major.minor`,
the protected baseline versions (numpy/torch/torchvision/torchaudio), and the
install-requirement hash. Installation is skipped **only when every one of those
fields matches**; any missing, legacy, or drifted field reinstalls and restarts.
An exactly matching marker always yields `PROCEED`, so there is no restart loop.

## DEPENDENCY HEALTH IS SCOPED TO S1

`pip check` audits the **entire** Colab image, which ships packages S1 never imports
(Gradio, IPython/jedi, ...). Its global verdict is recorded in full as a diagnostic but
is **not** a pass/fail gate. A conflict blocks the run only when the **complaining**
distribution is inside S1's own transitive import closure. The real NumPy
`RandomState` and dummy `torch.optim.AdamW` checks stay mandatory, and every module in
the S1 closure must import. NumPy/Torch are never reinstalled, and `huggingface_hub` is
never moved merely to satisfy a preinstalled Gradio.


## 1. Configuration (no scientific imports)


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = "https://github.com/vquclinh/MedNorm-VI.git"
REPO_REF = "main"

if "MEDNORM_DRIVE_ROOT" in os.environ:
    DRIVE_ROOT = Path(os.environ["MEDNORM_DRIVE_ROOT"])
if "MEDNORM_REPO_DIR" in os.environ:
    REPO_DIR = Path(os.environ["MEDNORM_REPO_DIR"])
if "MEDNORM_REPO_URL" in os.environ:
    REPO_URL = os.environ["MEDNORM_REPO_URL"]
if "MEDNORM_REPO_REF" in os.environ:
    REPO_REF = os.environ["MEDNORM_REPO_REF"]

CORPUS_DIR = (
    DRIVE_ROOT
    / "data"
    / "derived"
    / "training_corpora"
    / "mednorm_vi_training_v1"
)
# Artifact lifecycle: v1 recorded full_training_readiness: false and is immutable
# historical evidence. The corrected rerun writes to its OWN versioned directory
# and must never overwrite v1. The version is asserted against the tracked config.
SMOKE_ARTIFACT_VERSION = os.environ.get("MEDNORM_SMOKE_ARTIFACT_VERSION", "v3")
HISTORICAL_SMOKE_ARTIFACT_DIRS = (
    DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke",       # v1
    DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke_v2",    # v2
)
OUTPUT_DIR = DRIVE_ROOT / "artifacts" / f"s1_mention_first_run_smoke_{SMOKE_ARTIFACT_VERSION}"
for _historical in HISTORICAL_SMOKE_ARTIFACT_DIRS:
    assert OUTPUT_DIR.resolve() != _historical.resolve(), (
        f"the corrected smoke rerun must not overwrite the historical artifact {_historical}")
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoint"
TRAINING_MANIFEST_PATH = OUTPUT_DIR / "training_manifest.json"
SMOKE_CONFIG_REL = Path("configs/training/s1_mention_first_run_smoke.yaml")

FULL_TRAINING_ENABLED = False
CONFIRM_FULL_TRAINING = ""
if FULL_TRAINING_ENABLED or CONFIRM_FULL_TRAINING:
    raise SystemExit("This notebook is SMOKE_ONLY; use the separate full S1 workflow.")

IN_COLAB_BOOTSTRAP = "google.colab" in sys.modules
SEED = 20260723
print(json.dumps({
    "mode": "SMOKE_ONLY",
    "drive_root": str(DRIVE_ROOT),
    "repo_dir": str(REPO_DIR),
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "corpus_dir": str(CORPUS_DIR),
    "smoke_artifact_version": SMOKE_ARTIFACT_VERSION,
    "output_dir": str(OUTPUT_DIR),
    "historical_artifacts_preserved": [str(p) for p in HISTORICAL_SMOKE_ARTIFACT_DIRS],
}, indent=2, sort_keys=True))


## 2. Repository checkout (stdlib only; no NumPy/Torch import)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git",
    "clone",
    "--branch",
    REPO_REF,
    "--single-branch",
    REPO_URL,
    str(REPO_DIR),
], check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert len(RESOLVED_COMMIT) == 40 and all(c in "0123456789abcdef" for c in RESOLVED_COMMIT)
assert (REPO_DIR / "src" / "mednorm_vi").is_dir(), "cloned repo is missing src/mednorm_vi"
sys.path.insert(0, str(REPO_DIR / "src"))
print(json.dumps({"resolved_commit": RESOLVED_COMMIT, "src_verified": True}, indent=2, sort_keys=True))


## 3. Dependency metadata inspection (importlib.metadata; NumPy is NOT imported)


In [ ]:
import importlib.metadata as importlib_metadata

sys.path.insert(0, str(REPO_DIR / "src"))
from mednorm_vi.training.colab_bootstrap import (  # noqa: E402
    INSTALL_AND_RESTART,
    PROCEED,
    build_install_command,
    build_marker_fingerprint,
    build_pip_constraints,
    classify_dependency_health,
    compute_dependency_closure,
    decide_bootstrap_action,
    evaluate_full_training_readiness,
    load_dependency_contract,
    marker_mismatches,
    normalize_distribution_name,
    validate_abi_report,
    validate_install_command,
)

DEPENDENCY_CONTRACT_PATH = REPO_DIR / "configs" / "training" / "s1_mention_colab_dependencies.yaml"
contract = load_dependency_contract(DEPENDENCY_CONTRACT_PATH)
DEPENDENCY_CONTRACT_VERSION = contract.contract_version

TRACKED_PACKAGES = (
    "numpy", "torch", "torchvision", "torchaudio", "transformers", "tokenizers",
    "huggingface_hub", "sentencepiece", "accelerate", "py_vncorenlp", "scipy",
    "pandas", "scikit-learn", "safetensors", "pyarrow",
)

def package_version(name):
    """Version via metadata only - never imports the package (no NumPy load)."""
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return ""

baseline_versions = {name: package_version(name) for name in TRACKED_PACKAGES}
assert baseline_versions["numpy"], "no baseline NumPy detected in the Colab image"
assert baseline_versions["torch"], "no baseline Torch detected in the Colab image"

# The marker fingerprint binds a skipped installation to THIS tracked contract
# (exact file bytes), THIS Python major.minor, THIS protected baseline, and THIS
# normalized requirement set. A version string alone is far too weak.
PYTHON_MAJOR_MINOR = f"{sys.version_info.major}.{sys.version_info.minor}"
MARKER_FINGERPRINT = build_marker_fingerprint(
    contract, PYTHON_MAJOR_MINOR, baseline_versions)
DEPENDENCY_CONTRACT_SHA256 = contract.contract_sha256
INSTALL_REQUIREMENT_HASH = contract.install_requirement_hash
print(json.dumps({
    "baseline_versions": {k: v for k, v in baseline_versions.items() if v},
    "marker_fingerprint": MARKER_FINGERPRINT.as_dict(),
}, indent=2, sort_keys=True))


## 4. Consolidated installation + forced kernel restart (PASS 1 only)


In [ ]:
MARKER_PATH = Path(contract.marker_path)
CONSTRAINT_PATH = Path("/content/mednorm_s1_constraints.txt")

bootstrap_marker = None
if MARKER_PATH.is_file():
    try:
        bootstrap_marker = json.loads(MARKER_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        bootstrap_marker = None

# Every fingerprint field must match before installation may be skipped.
BOOTSTRAP_MARKER_MISMATCHES = marker_mismatches(bootstrap_marker, MARKER_FINGERPRINT)
BOOTSTRAP_ACTION = decide_bootstrap_action(bootstrap_marker, MARKER_FINGERPRINT)
print(json.dumps({
    "bootstrap_action": BOOTSTRAP_ACTION,
    "marker_path": str(MARKER_PATH),
    "marker_mismatches": BOOTSTRAP_MARKER_MISMATCHES,
    "expected_fingerprint": MARKER_FINGERPRINT.as_dict(),
}, indent=2, sort_keys=True))

if BOOTSTRAP_ACTION == INSTALL_AND_RESTART:
    # Pin the INHERITED stack to the versions this runtime already provides so pip
    # cannot silently move NumPy/Torch. An incompatible request now fails loudly
    # instead of corrupting the C-ABI.
    constraints = build_pip_constraints(baseline_versions)
    CONSTRAINT_PATH.write_text("\n".join(constraints) + "\n", encoding="utf-8")
    install_command = build_install_command(contract, str(CONSTRAINT_PATH), sys.executable)
    validate_install_command(install_command)
    print("constraints:", constraints)
    print("install:", " ".join(install_command))
    subprocess.run(install_command, check=True)
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
    pip_check_output = "\n".join(
        part for part in (pip_check.stdout.strip(), pip_check.stderr.strip()) if part)
    # Written with the CURRENT fingerprint, so PASS 2 matches exactly and the
    # notebook can never restart twice for the same environment.
    MARKER_PATH.write_text(json.dumps({
        "install_completed": True,
        **MARKER_FINGERPRINT.as_dict(),
        "baseline_versions": baseline_versions,
        "constraints": constraints,
        "installed_specifiers": list(contract.install_specifiers),
        "pip_check_returncode": pip_check.returncode,
        "pip_check_stdout": pip_check.stdout,
        "pip_check_stderr": pip_check.stderr,
        "pip_check_output": pip_check_output,
    }, indent=2, sort_keys=True), encoding="utf-8")
    print("=" * 78)
    print("PASS 1 COMPLETE - the kernel is about to restart (this is expected).")
    print("AFTER the restart finishes, run Runtime > Run all AGAIN to execute PASS 2.")
    print("=" * 78)
    if IN_COLAB_BOOTSTRAP:
        os.kill(os.getpid(), 9)  # real kernel restart; Colab reconnects automatically
    else:
        raise SystemExit("dependency installation requires a kernel restart")
else:
    print("PASS 2: bootstrap marker matches every fingerprint field;",
          DEPENDENCY_CONTRACT_VERSION, DEPENDENCY_CONTRACT_SHA256[:16])

DEPENDENCY_RESTART_COMPLETED = BOOTSTRAP_ACTION == PROCEED


## 5. Post-restart dependency verification


In [ ]:
assert DEPENDENCY_RESTART_COMPLETED, (
    "PASS 1 installs dependencies and restarts the kernel. Run all cells again for PASS 2.")

# Re-validate the marker in the restarted kernel: PASS 1 wrote it in a different
# process, so the fingerprint is re-checked here against the live runtime.
POST_RESTART_MARKER_MISMATCHES = marker_mismatches(bootstrap_marker, MARKER_FINGERPRINT)
assert not POST_RESTART_MARKER_MISMATCHES, (
    f"bootstrap marker no longer matches this runtime: {POST_RESTART_MARKER_MISMATCHES}")

installed_versions = {name: package_version(name) for name in TRACKED_PACKAGES}
marker_baseline = dict(bootstrap_marker.get("baseline_versions", {}))
changed_packages = {
    name: {"baseline": marker_baseline.get(name, ""), "current": installed_versions[name]}
    for name in TRACKED_PACKAGES
    if marker_baseline.get(name, "") != installed_versions[name]
}
protected_changed = {
    name: change for name, change in changed_packages.items()
    if name in ("numpy", "torch", "torchvision", "torchaudio")
}
assert not protected_changed, (
    f"inherited stack was modified despite constraints: {protected_changed}")
# `pip check` is captured IN FULL (stdout and stderr, never truncated) and kept as a
# diagnostic. It audits the whole Colab image, so its global verdict alone must not
# gate S1: preinstalled Gradio/IPython complaints are unrelated to this smoke.
pip_check_proc = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
PIP_CHECK_OUTPUT = "\n".join(
    part for part in (pip_check_proc.stdout.strip(), pip_check_proc.stderr.strip()) if part)
PIP_CHECK_PASSED = pip_check_proc.returncode == 0

def installed_requirement_graph():
    """Requirement graph from metadata only - imports nothing (no NumPy load)."""
    graph = {}
    for dist in importlib_metadata.distributions():
        dist_name = normalize_distribution_name(dist.metadata["Name"] or "")
        if dist_name:
            graph.setdefault(dist_name, []).extend(dist.requires or [])
    return graph

# The closure is what S1 ACTUALLY depends on: the contract's import roots plus their
# transitive requirements, resolved from the installed metadata of this runtime.
S1_DEPENDENCY_CLOSURE = compute_dependency_closure(
    contract.closure_root_distributions, installed_requirement_graph())
DEPENDENCY_HEALTH = classify_dependency_health(
    PIP_CHECK_OUTPUT, S1_DEPENDENCY_CLOSURE, pip_check_proc.returncode)
print("pip check output (complete):")
print(PIP_CHECK_OUTPUT or "(no broken requirements reported)")
print(json.dumps({
    "bootstrap_action": BOOTSTRAP_ACTION,
    "marker_mismatches": POST_RESTART_MARKER_MISMATCHES,
    "dependency_contract_sha256": DEPENDENCY_CONTRACT_SHA256,
    "install_requirement_hash": INSTALL_REQUIREMENT_HASH,
    "python_major_minor": PYTHON_MAJOR_MINOR,
    "changed_packages": changed_packages,
    "protected_stack_unchanged": True,
    "pip_check_passed": PIP_CHECK_PASSED,
    "s1_dependency_closure_size": len(S1_DEPENDENCY_CLOSURE),
    "s1_dependency_healthy": DEPENDENCY_HEALTH.healthy,
    "blocking_dependency_conflicts": [c.message for c in DEPENDENCY_HEALTH.blocking],
    "non_blocking_dependency_conflicts": [c.message for c in DEPENDENCY_HEALTH.non_blocking],
}, indent=2, sort_keys=True))
if DEPENDENCY_HEALTH.non_blocking:
    print("NOTE: the conflicts above are outside the S1 dependency closure and are")
    print("      recorded as non-blocking diagnostics. They are NOT remediated here:")
    print("      huggingface_hub is not upgraded for Gradio, and NumPy/Torch are untouched.")


## 6. NumPy / AdamW ABI preflight (fail fast before any acquisition)


In [ ]:
# FAIL-FAST NumPy/Torch C-ABI health. This is the FIRST place NumPy or Torch is
# imported, and it runs BEFORE any Drive mount, corpus, VnCoreNLP, tokenizer, or
# model acquisition. The Audit 0022 run died here in disguise: the ABI was already
# broken, and torch.optim.AdamW merely triggered the first compiled numpy.random import.
abi_report = {
    "numpy_imported": False,
    "numpy_random_imported": False,
    "torch_imported": False,
    "adamw_constructed": False,
    "pip_check_passed": PIP_CHECK_PASSED,
}
try:
    import numpy as np  # noqa: E402
    abi_report["numpy_imported"] = True

    from numpy.random import RandomState  # noqa: E402

    rng = RandomState(42)
    values = rng.rand(4)
    assert values.shape == (4,)
    abi_report["numpy_random_imported"] = True

    import numpy.random.mtrand as numpy_mtrand  # noqa: E402

    abi_report.update({
        "numpy_version": np.__version__,
        "numpy_path": str(Path(np.__file__).resolve().parent),
        "numpy_mtrand_path": str(Path(numpy_mtrand.__file__).resolve()),
    })
    # numpy.core is a DEPRECATED NumPy 2.x compatibility shim; it is a diagnostic
    # path only, so its absence must never be read as an ABI failure.
    try:
        import numpy.core as numpy_core  # noqa: E402

        abi_report["numpy_core_path"] = str(Path(numpy_core.__file__).resolve().parent)
    except Exception as core_exc:  # noqa: BLE001 - diagnostic only
        abi_report["numpy_core_path"] = f"unavailable: {type(core_exc).__name__}"

    import torch  # noqa: E402

    abi_report["torch_imported"] = True
    abi_report.update({
        "torch_version": torch.__version__,
        "torch_path": str(Path(torch.__file__).resolve().parent),
        "cuda_available": bool(torch.cuda.is_available()),
    })

    dummy_parameter = torch.nn.Parameter(torch.zeros(1))
    dummy_optimizer = torch.optim.AdamW([dummy_parameter], lr=1e-3)
    dummy_optimizer.zero_grad(set_to_none=True)
    abi_report["adamw_constructed"] = True

    # Validate the ACTUAL S1 dependency closure: every module S1 imports must load.
    # This is the positive check that replaces a global pip check verdict.
    s1_import_failures = []
    s1_import_versions = {}
    for _distribution, _module in contract.import_closure_roots:
        try:
            _imported = importlib.import_module(_module)
            s1_import_versions[_module] = str(
                getattr(_imported, "__version__", "") or package_version(_distribution))
        except Exception as import_exc:  # noqa: BLE001 - collect every failure
            s1_import_failures.append(f"{_module}: {type(import_exc).__name__}: {import_exc}")
    abi_report["s1_import_failures"] = s1_import_failures
    abi_report["s1_import_versions"] = s1_import_versions
    abi_report["s1_dependency_closure_verified"] = True
except Exception as exc:  # noqa: BLE001 - diagnostics then fail fast
    abi_report["error"] = f"{type(exc).__name__}: {exc}"
    print(json.dumps({
        "abi_preflight": "FAILED",
        "report": abi_report,
        "python": platform.python_version(),
        "sys_path_head": sys.path[:5],
        "pythonpath": os.environ.get("PYTHONPATH", ""),
    }, indent=2, sort_keys=True))
    raise

abi_report["numpy_distribution_count"] = sum(
    1 for dist in importlib_metadata.distributions()
    if (dist.metadata["Name"] or "").lower() == "numpy")
abi_report["python_version"] = platform.python_version()
# Closure-scoped dependency health. `pip_check_output` is carried in full; only
# conflicts raised BY the S1 closure can produce a blocking problem.
abi_report.update(DEPENDENCY_HEALTH.as_dict())

abi_problems = validate_abi_report(abi_report)
assert not abi_problems, f"NumPy/Torch ABI preflight failed: {abi_problems}"
NUMPY_ABI_PREFLIGHT_PASSED = True
S1_DEPENDENCY_CLOSURE_VERIFIED = True
print(json.dumps({"abi_preflight": "PASSED", "report": abi_report}, indent=2, sort_keys=True))


## 7. Runtime, GPU, and Drive mount


In [ ]:
runtime_report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "in_colab": IN_COLAB_BOOTSTRAP,
    "dependency_contract_version": DEPENDENCY_CONTRACT_VERSION,
    "dependency_restart_completed": DEPENDENCY_RESTART_COMPLETED,
    "numpy_abi_preflight_passed": NUMPY_ABI_PREFLIGHT_PASSED,
}
assert IN_COLAB_BOOTSTRAP, "S1 smoke must run on Google Colab Pro with a GPU."
assert torch.cuda.is_available(), "S1 smoke requires a Colab GPU runtime."
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
runtime_report.update({
    "torch": torch.__version__,
    "cuda_available": True,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_vram_gb": round(props.total_memory / 1e9, 2),
})

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

drive_module = importlib.import_module("google.colab.drive")
drive_module.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(runtime_report, indent=2, sort_keys=True))


## 8. Corpus gate before model acquisition


In [ ]:
from mednorm_vi.model_registry.registry import load_registry, validate_profile_budget
from mednorm_vi.training.phobert_alignment import (
    ALIGNMENT_BACKEND,
    looks_pre_segmented,
    resolve_segmented_text,
    BOUNDARY_MERGE_POLICY,
    STAGE_SUBTOKEN_ENCODING,
    STAGE_TOKENIZER_EQUIVALENCE,
    STAGE_WORD_MAPPING,
    PARTIAL_TRUNCATION_POLICY,
    SUBTOKEN_SUPERVISION_POLICY,
    AlignmentError,
    describe_backend,
    map_segmented_words,
    segmented_text_to_words,
    verify_tokenizer_equivalence,
)
from mednorm_vi.training.s1_mention_smoke import (
    ENTITY_TYPE_ORDER,
    smoke_artifact_paths_from_config,
    alignment_diagnostic,
    governed_exclusion_diagnostic,
    load_governed_exclusions,
    privacy_safe_example_id,
    summarize_alignment_diagnostics,
    encode_mention_example_slow,
    expected_corpus_from_config,
    load_coverage,
    load_smoke_config,
    pad_encoded_features,
    preparation_digest,
    select_deterministic_examples,
    sha256_file,
    smoke_limits_from_config,
    verify_governed_corpus,
    verify_vietmed_train_only_masks,
)

smoke_config = load_smoke_config(REPO_DIR / SMOKE_CONFIG_REL)
expected_corpus = expected_corpus_from_config(smoke_config)
limits = smoke_limits_from_config(smoke_config)
print(json.dumps({
    "max_train_examples": limits.max_train_examples,
    "max_validation_examples": limits.max_validation_examples,
    "max_train_batches": limits.max_train_batches,
    "max_validation_batches": limits.max_validation_batches,
    "max_optimizer_steps": limits.max_optimizer_steps,
    "batch_size": limits.batch_size,
    "max_sequence_length": limits.max_sequence_length,
}, indent=2, sort_keys=True))

# The output path is tracked in configuration, not only in this notebook.
smoke_artifact_paths = smoke_artifact_paths_from_config(smoke_config)
assert smoke_artifact_paths.artifact_version == SMOKE_ARTIFACT_VERSION, (
    f"notebook version {SMOKE_ARTIFACT_VERSION!r} != tracked config "
    f"{smoke_artifact_paths.artifact_version!r}")
assert Path(smoke_artifact_paths.artifact_dir).name == OUTPUT_DIR.name, (
    f"tracked artifact_dir {smoke_artifact_paths.artifact_dir!r} != {str(OUTPUT_DIR)!r}")
print(json.dumps(smoke_artifact_paths.as_dict(), indent=2, sort_keys=True))

corpus_report = verify_governed_corpus(CORPUS_DIR, expected_corpus)
print(json.dumps(corpus_report, indent=2, sort_keys=True))


## 9. Registry, parameter budget, and smoke model source


In [ ]:
model_cfg = smoke_config["model"]
roles = load_registry(REPO_DIR / "configs" / "model_registry" / "models_v1.yaml")
role = next(r for r in roles if r.model_id == model_cfg["registry_model_id"])
profile_budget = validate_profile_budget(roles, profile="full")
assert profile_budget.within_9b, "full model profile exceeds the 9B budget"
MODEL_ID = str(model_cfg["hf_model_id"])
MODEL_REVISION = str(model_cfg["revision"])
model_registry_report = {
    "registry_model_id": role.model_id,
    "role": role.role,
    "approved_model_source": model_cfg["approved_source"],
    "hf_model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "registry_base_parameters": role.base_parameter_count,
    "registry_adapter_parameters": role.adapter_parameter_count,
    "full_profile_base_parameters": profile_budget.base_parameters,
    "full_profile_adapter_parameters": profile_budget.adapter_parameters,
    "full_profile_total_parameters": profile_budget.total_parameters,
    "full_profile_within_9b": profile_budget.within_9b,
}
print(json.dumps(model_registry_report, indent=2, sort_keys=True))


## 10. Deterministic smoke subsets and loss masks


In [ ]:
coverage_by_source = load_coverage(CORPUS_DIR)
train_path = CORPUS_DIR / "splits" / "train.jsonl"
validation_path = CORPUS_DIR / "splits" / "validation.jsonl"
vietmed_train_examples = select_deterministic_examples(
    train_path,
    limit=1,
    seed=SEED,
    require_entities=True,
    source_dataset="vietmed_ner",
)
general_train_examples = select_deterministic_examples(
    train_path,
    limit=limits.max_train_examples,
    seed=SEED,
    require_entities=True,
)
seen_ids = set()
train_examples = []
for row in [*vietmed_train_examples, *general_train_examples]:
    if row["example_id"] not in seen_ids:
        train_examples.append(row)
        seen_ids.add(row["example_id"])
train_examples = train_examples[:limits.max_train_examples]
validation_examples = select_deterministic_examples(
    validation_path,
    limit=limits.max_validation_examples,
    seed=SEED,
    require_entities=True,
)
mask_report = verify_vietmed_train_only_masks(train_examples, coverage_by_source)
assert mask_report["vietmed_examples"] >= 1
assert mask_report["vietmed_entities"] >= 1
subset_report = {
    "train_examples": len(train_examples),
    "validation_examples": len(validation_examples),
    "vietmed_mask_report": mask_report,
    "train_split_sha256": sha256_file(train_path),
}
print(json.dumps(subset_report, indent=2, sort_keys=True))


## 11. Transformers import (post-ABI; no installation here)


In [ ]:
# Already imported and version-checked by the S1 closure preflight; no install here.
transformers_module = importlib.import_module("transformers")
AutoModel = transformers_module.AutoModel
AutoTokenizer = transformers_module.AutoTokenizer
runtime_report["transformers"] = transformers_module.__version__
print(json.dumps({
    "transformers": transformers_module.__version__,
    "tokenizers": package_version("tokenizers"),
    "huggingface_hub": package_version("huggingface_hub"),
}, indent=2, sort_keys=True))


## 12. Word segmentation contract (VnCoreNLP RDRSegmenter required)


In [ ]:
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"
# Production S1 smoke REQUIRES VnCoreNLP RDRSegmenter. whitespace_fallback is an
# explicit opt-in diagnostic mode only; it never activates automatically.
SEGMENTER_MODE = os.environ.get("MEDNORM_SEGMENTER_MODE", "vncorenlp")
assert SEGMENTER_MODE in ("vncorenlp", "whitespace_fallback"), SEGMENTER_MODE
DEGRADED_FALLBACK = SEGMENTER_MODE == "whitespace_fallback"

segmenter_report = {
    "segmenter_mode": SEGMENTER_MODE,
    "word_segmenter": "",
    "word_segmenter_version": "",
    "word_segmenter_resource_hashes": {},
    "degraded_fallback": DEGRADED_FALLBACK,
    "resource_dir": str(VNCORENLP_DIR),
    "acquisition_source": "",
}

if SEGMENTER_MODE == "vncorenlp":
    # py_vncorenlp was installed in the single constrained transaction (PASS 1).
    VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
    py_vncorenlp = importlib.import_module("py_vncorenlp")
    if not any(VNCORENLP_DIR.glob("*.jar")):
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
    _resources = sorted(
        [q for q in VNCORENLP_DIR.rglob("*") if q.is_file()], key=lambda q: q.name)
    assert _resources, "VnCoreNLP resources missing after acquisition (fail fast)"
    _hashes = {q.name: sha256_file(q) for q in _resources if q.suffix in (".jar", ".xz", ".txt")}
    assert _hashes, "VnCoreNLP resource hashes are empty (broken installation; fail fast)"
    _rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
    segmenter_report.update({
        "word_segmenter": "VnCoreNLP RDRSegmenter",
        "word_segmenter_version": "py_vncorenlp==0.1.4",
        "word_segmenter_resource_hashes": _hashes,
        "acquisition_source": "py_vncorenlp.download_model",
    })

    def _run_segmenter(text):
        segments = _rdr.word_segment(text)
        assert segments, "RDRSegmenter returned no segments (fail fast)"
        return " ".join(segments)
else:
    print("=" * 78)
    print("!! DEGRADED MODE: whitespace_fallback is NOT the production S1 path.")
    print("!! Word segmentation does not match ViHealthBERT-Word pretraining.")
    print("!! This run cannot be classified as a successful production-path S1 smoke.")
    print("=" * 78)
    segmenter_report.update({
        "word_segmenter": "whitespace (degraded diagnostics only)",
        "word_segmenter_version": "builtin-whitespace",
        "acquisition_source": "none (degraded diagnostics mode)",
    })

    def _run_segmenter(text):
        return " ".join(text.split())


# SEGMENTATION POLICY (single rule for raw and pre-segmented sources):
# text that is ALREADY RDRSegmenter output is used verbatim - re-segmenting it
# splits the join character off as a standalone token and shreds the very words
# ViHealthBERT-Word expects. Everything else goes through the production segmenter.
def segment_example_text(text):
    segmented, _source = resolve_segmented_text(text, _run_segmenter)
    return segmented


segmenter_report["segmentation_policy"] = (
    "pre_segmented_source_used_verbatim_else_vncorenlp")

PRODUCTION_SEGMENTATION = (
    segmenter_report["segmenter_mode"] == "vncorenlp"
    and segmenter_report["word_segmenter"] == "VnCoreNLP RDRSegmenter"
    and segmenter_report["degraded_fallback"] is False
    and bool(segmenter_report["word_segmenter_resource_hashes"])
)
print(json.dumps({k: v for k, v in segmenter_report.items()
                  if k != "word_segmenter_resource_hashes"}, indent=2, sort_keys=True))
print("resource_hash_count", len(segmenter_report["word_segmenter_resource_hashes"]))
print("production_segmentation", PRODUCTION_SEGMENTATION)


## 13. Slow tokenizer acquisition


In [ ]:
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(MODEL_CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ViHealthBERT-Word declares tokenizer_class = PhobertTokenizer, which has NO fast
# implementation: requesting a fast tokenizer silently returns the slow one and
# tokenizer.is_fast stays False. Load it honestly as slow and align manually.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
    use_fast=False,
)
tokenizer_report = describe_backend(tokenizer)
tokenizer_report.update({
    "requested_revision": MODEL_REVISION,
    "pad_token_id": tokenizer.pad_token_id,
    "cls_token_id": getattr(tokenizer, "cls_token_id", None),
    "sep_token_id": getattr(tokenizer, "sep_token_id", None),
    "unk_token_id": getattr(tokenizer, "unk_token_id", None),
})
assert tokenizer_report["tokenizer_is_fast"] is False, (
    "expected the slow PhobertTokenizer; a fast tokenizer would change the alignment contract")
print(json.dumps(tokenizer_report, indent=2, sort_keys=True))


## 14. Tokenizer equivalence preflight (before model weights)


In [ ]:
# Alignment/equivalence diagnostics are PRIVACY-SAFE: dataset, split, a hashed
# example id, the pipeline stage, a reason code, and the exception class. Never
# raw clinical text, raw entity text, or a verbatim example id.
GOVERNED_EXCLUSIONS = load_governed_exclusions(
    REPO_DIR / "configs" / "training" / "s1_governed_exclusions.yaml")
alignment_diagnostics = []
SMOKE_SPLITS = (("train", train_examples), ("validation", validation_examples))
EXAMPLES_CONSIDERED = sum(len(rows) for _, rows in SMOKE_SPLITS)

# SCOPE: tokenizer equivalence and the alignment preflight now cover the SAME
# examples (train + validation), so their counters reconcile. Previously
# equivalence ran over both splits while the alignment preflight ran over train
# only, and validation failures were silently dropped (Audit 0026).
tokenizer_equivalence = {
    "tokenizer_equivalence_checked": True,
    "tokenizer_equivalence_scope": [name for name, _ in SMOKE_SPLITS],
    "tokenizer_equivalence_considered": EXAMPLES_CONSIDERED,
    "tokenizer_equivalence_examples": 0,
    "tokenizer_equivalence_failures": 0,
    "tokenizer_equivalence_skipped_unmappable": 0,
}
mapped_words_by_example = {}
_equivalence_errors = []
for split_name, rows in SMOKE_SPLITS:
    for row in rows:
        handle = privacy_safe_example_id(row["example_id"])
        if handle in GOVERNED_EXCLUSIONS:
            alignment_diagnostics.append(
                governed_exclusion_diagnostic(row, split=split_name))
            continue
        try:
            _words = map_segmented_words(
                row["text"], segmented_text_to_words(segment_example_text(row["text"])))
        except AlignmentError as exc:
            # Word mapping failed: equivalence is undefined for this example, and
            # the failure is recorded instead of silently skipped.
            tokenizer_equivalence["tokenizer_equivalence_skipped_unmappable"] += 1
            alignment_diagnostics.append(alignment_diagnostic(
                row, split=split_name, stage=STAGE_WORD_MAPPING, error=exc))
            continue
        mapped_words_by_example[row["example_id"]] = _words
        try:
            verify_tokenizer_equivalence(_words, tokenizer)
            tokenizer_equivalence["tokenizer_equivalence_examples"] += 1
        except AlignmentError as exc:
            tokenizer_equivalence["tokenizer_equivalence_failures"] += 1
            _equivalence_errors.append(str(exc))
            alignment_diagnostics.append(alignment_diagnostic(
                row, split=split_name, stage=STAGE_TOKENIZER_EQUIVALENCE, error=exc))

assert tokenizer_equivalence["tokenizer_equivalence_failures"] == 0, (
    "per-word alignment does not match whole-sentence tokenization: "
    + "; ".join(_equivalence_errors[:3]))
assert tokenizer_equivalence["tokenizer_equivalence_examples"] > 0, (
    "no example passed tokenizer equivalence (fail fast)")
print(json.dumps(tokenizer_equivalence, indent=2, sort_keys=True))

## 15. Alignment preflight (before model weights)


In [ ]:
alignment_preflight = {
    "alignment_scope": [name for name, _ in SMOKE_SPLITS],
    "examples_considered": EXAMPLES_CONSIDERED,
    "aligned_example_count": 0,
    "truncated_example_count": 0,
    "truncated_entity_count": 0,
    "fully_dropped_entity_count": 0,
    "partially_truncated_entity_count": 0,
    "boundary_merge_masked_word_count": 0,
    "boundary_merge_affected_entity_count": 0,
    "pre_segmented_example_count": 0,
    "segmenter_example_count": 0,
    "supervised_token_count": 0,
    "positive_label_count": 0,
}
encoded_by_split = {"train": [], "validation": []}
already_failed = {d.privacy_safe_example_id for d in alignment_diagnostics}
for split_name, rows in SMOKE_SPLITS:
    for row in rows:
        if privacy_safe_example_id(row["example_id"]) in already_failed:
            continue          # word mapping or governed exclusion already recorded it
        try:
            feature = encode_mention_example_slow(
                row,
                tokenizer,
                coverage_by_source=coverage_by_source,
                max_length=limits.max_sequence_length,
                segmented_text=segment_example_text(row["text"]),
            )
        except (AlignmentError, ValueError) as exc:
            alignment_diagnostics.append(alignment_diagnostic(
                row, split=split_name, stage=STAGE_SUBTOKEN_ENCODING, error=exc))
            continue
        assert len(feature["input_ids"]) == len(feature["attention_mask"]) == len(feature["labels"]) == len(feature["label_mask"])
        alignment_preflight["aligned_example_count"] += 1
        if looks_pre_segmented(row["text"]):
            alignment_preflight["pre_segmented_example_count"] += 1
        else:
            alignment_preflight["segmenter_example_count"] += 1
        alignment_preflight["truncated_example_count"] += int(bool(feature["truncated"]))
        alignment_preflight["truncated_entity_count"] += int(feature["truncated_entity_count"])
        alignment_preflight["fully_dropped_entity_count"] += int(feature["fully_dropped_entity_count"])
        alignment_preflight["partially_truncated_entity_count"] += int(feature["partially_truncated_entity_count"])
        alignment_preflight["boundary_merge_masked_word_count"] += int(feature["boundary_merge_masked_word_count"])
        alignment_preflight["boundary_merge_affected_entity_count"] += int(feature["boundary_merge_affected_entity_count"])
        alignment_preflight["supervised_token_count"] += sum(feature["label_mask"])
        alignment_preflight["positive_label_count"] += sum(1 for lab in feature["labels"] if any(lab))
        encoded_by_split[split_name].append(feature)

# Unexpected failures block readiness; tracked governed exclusions do not.
alignment_preflight.update(summarize_alignment_diagnostics(alignment_diagnostics))
alignment_preflight["boundary_merge_policy"] = BOUNDARY_MERGE_POLICY

# Reconciliation invariant: every considered example is accounted for exactly once.
RECONCILED = (
    alignment_preflight["aligned_example_count"]
    + alignment_preflight["unalignable_example_count"]
    + alignment_preflight["governed_exclusion_count"]
) == EXAMPLES_CONSIDERED
alignment_preflight["counters_reconciled"] = RECONCILED
assert RECONCILED, f"alignment counters do not reconcile: {alignment_preflight}"

encoded_train = encoded_by_split["train"]
encoded_validation = encoded_by_split["validation"]
assert encoded_train, "alignment preflight produced no usable training features"
assert encoded_validation, "alignment preflight produced no usable validation features"
if alignment_preflight["unalignable_examples"]:
    print("UNEXPECTED unalignable examples (privacy-safe diagnostics):")
    for entry in alignment_preflight["unalignable_examples"]:
        print("  ", json.dumps(entry, sort_keys=True))
print(json.dumps(alignment_preflight, indent=2, sort_keys=True))

## 16. Load the backbone (after all preflights passed)


In [ ]:
backbone = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
).to(device)
resolved_model_revision = getattr(backbone.config, "_commit_hash", "") or MODEL_REVISION
base_parameter_count_actual = sum(p.numel() for p in backbone.parameters())
print(json.dumps({
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "resolved_model_revision": resolved_model_revision,
    "actual_base_parameters": base_parameter_count_actual,
    "registry_base_parameters": role.base_parameter_count,
    "cache_dir": str(MODEL_CACHE_DIR),
}, indent=2, sort_keys=True))


## 17. Dataset encoding and collation


In [ ]:
# encoded_train / encoded_validation were built by the alignment preflight above
# (slow-tokenizer path). Re-encode the train subset to prove determinism.
encoded_train_repeat = []
for row in train_examples:
    try:
        encoded_train_repeat.append(encode_mention_example_slow(
            row, tokenizer, coverage_by_source=coverage_by_source,
            max_length=limits.max_sequence_length,
            segmented_text=segment_example_text(row["text"]),
        ))
    except (AlignmentError, ValueError):
        continue
preparation_hash = preparation_digest(encoded_train)
assert preparation_hash == preparation_digest(encoded_train_repeat), "fixed-seed preparation is not deterministic"
batch = pad_encoded_features(
    encoded_train[:limits.batch_size],
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1,
)
batch_shapes = {
    "batch_size": len(batch["input_ids"]),
    "sequence_length": len(batch["input_ids"][0]),
    "entity_type_count": len(ENTITY_TYPE_ORDER),
    "preparation_hash": preparation_hash,
}
assert all(len(r) == batch_shapes["sequence_length"] for r in batch["attention_mask"])
assert all(len(r) == batch_shapes["sequence_length"] for r in batch["label_mask"])
print(json.dumps(batch_shapes, indent=2, sort_keys=True))


## 18. One bounded forward, backward, and optimizer step


In [ ]:
class MentionTokenClassifier(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, label_count: int) -> None:
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(hidden_size, label_count)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(output.last_hidden_state))

model = MentionTokenClassifier(backbone, len(ENTITY_TYPE_ORDER)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=limits.learning_rate)
criterion = torch.nn.BCEWithLogitsLoss(reduction="none")
trainable_parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
model.train()
input_ids = torch.tensor(batch["input_ids"], dtype=torch.long, device=device)
attention_mask = torch.tensor(batch["attention_mask"], dtype=torch.long, device=device)
labels = torch.tensor(batch["labels"], dtype=torch.float32, device=device)
label_mask = torch.tensor(batch["label_mask"], dtype=torch.float32, device=device)
optimizer.zero_grad(set_to_none=True)
logits = model(input_ids=input_ids, attention_mask=attention_mask)
loss_by_token = criterion(logits, labels)
normalizer = torch.clamp(label_mask.sum() * len(ENTITY_TYPE_ORDER), min=1.0)
loss = (loss_by_token * label_mask.unsqueeze(-1)).sum() / normalizer
assert torch.isfinite(loss).item(), "smoke loss is not finite"
loss.backward()
optimizer.step()
optimizer_step_confirmed = True
TRAIN_LOSS_FINITE = bool(torch.isfinite(loss).item())
BACKWARD_COMPLETED = any(
    param.grad is not None for param in model.parameters() if param.requires_grad)
OPTIMIZER_STEP_COMPLETED = bool(optimizer_step_confirmed)
assert TRAIN_LOSS_FINITE and BACKWARD_COMPLETED and OPTIMIZER_STEP_COMPLETED
loss_values = {"train_loss": float(loss.detach().cpu()), "optimizer_step_confirmed": optimizer_step_confirmed,
               "train_loss_finite": TRAIN_LOSS_FINITE,
               "backward_completed": BACKWARD_COMPLETED}
print(json.dumps({
    "batch_shapes": batch_shapes,
    "loss_values": loss_values,
    "trainable_parameter_count": trainable_parameter_count,
}, indent=2, sort_keys=True))


## 19. Tiny validation inference


In [ ]:
model.eval()
validation_losses = []
positive_logits = 0
validation_batches_used = 0
with torch.no_grad():
    for start in range(0, min(len(encoded_validation), limits.batch_size * limits.max_validation_batches), limits.batch_size):
        features = encoded_validation[start:start + limits.batch_size]
        if not features:
            continue
        vb = pad_encoded_features(
            features,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0,
        )
        vi = torch.tensor(vb["input_ids"], dtype=torch.long, device=device)
        va = torch.tensor(vb["attention_mask"], dtype=torch.long, device=device)
        vl = torch.tensor(vb["labels"], dtype=torch.float32, device=device)
        vm = torch.tensor(vb["label_mask"], dtype=torch.float32, device=device)
        out = model(input_ids=vi, attention_mask=va)
        raw = criterion(out, vl)
        denom = torch.clamp(vm.sum() * len(ENTITY_TYPE_ORDER), min=1.0)
        vloss = (raw * vm.unsqueeze(-1)).sum() / denom
        assert torch.isfinite(vloss).item(), "validation smoke loss is not finite"
        validation_losses.append(float(vloss.cpu()))
        positive_logits += int(((torch.sigmoid(out) > 0.5) * vm.unsqueeze(-1)).sum().cpu())
        validation_batches_used += 1
VALIDATION_COMPLETED = validation_batches_used >= 1 and all(
    isinstance(v, float) for v in validation_losses)
assert VALIDATION_COMPLETED, "validation smoke did not complete"
validation_metrics = {
    "validation_batches": validation_batches_used,
    "validation_examples": min(len(encoded_validation), limits.batch_size * limits.max_validation_batches),
    "mean_validation_loss": sum(validation_losses) / max(1, len(validation_losses)),
    "positive_token_type_predictions": positive_logits,
    "validation_completed": VALIDATION_COMPLETED,
}
print(json.dumps(validation_metrics, indent=2, sort_keys=True))


## 20. Checkpoint save, reload, and manifest


In [ ]:
checkpoint_path = CHECKPOINT_DIR / "s1_mention_smoke_model.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "entity_type_order": ENTITY_TYPE_ORDER,
    "mode": "SMOKE_ONLY",
    "seed": SEED,
}, checkpoint_path)
checkpoint_sha256 = sha256_file(checkpoint_path)
loaded_checkpoint = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(loaded_checkpoint["model_state_dict"])
assert loaded_checkpoint["mode"] == "SMOKE_ONLY"
training_manifest = {
    "manifest_version": 1,
    "stage_id": "S1",
    "role": "mention/vihealthbert",
    "status": "SMOKE_ONLY",
    "smoke_only_not_full_training": True,
    "full_training_readiness": evaluate_full_training_readiness({
        "production_segmentation": PRODUCTION_SEGMENTATION,
        "tokenizer_equivalence_examples": tokenizer_equivalence["tokenizer_equivalence_examples"],
        "tokenizer_equivalence_failures": tokenizer_equivalence["tokenizer_equivalence_failures"],
        "unalignable_example_count": alignment_preflight["unalignable_example_count"],
        "dependency_restart_completed": DEPENDENCY_RESTART_COMPLETED,
        "numpy_abi_preflight_passed": NUMPY_ABI_PREFLIGHT_PASSED,
        "s1_dependency_closure_verified": S1_DEPENDENCY_CLOSURE_VERIFIED,
        "train_loss_finite": TRAIN_LOSS_FINITE,
        "backward_completed": BACKWARD_COMPLETED,
        "optimizer_step_completed": OPTIMIZER_STEP_COMPLETED,
        "validation_completed": VALIDATION_COMPLETED,
        "checkpoint_saved": True,
        "checkpoint_reloaded": True,
    }),
    "environment": {
        "dependency_contract_version": DEPENDENCY_CONTRACT_VERSION,
        "dependency_contract_sha256": DEPENDENCY_CONTRACT_SHA256,
        "install_requirement_hash": INSTALL_REQUIREMENT_HASH,
        "python_major_minor": PYTHON_MAJOR_MINOR,
        "protected_baseline_versions": MARKER_FINGERPRINT.protected_baseline_versions,
        "bootstrap_action": BOOTSTRAP_ACTION,
        "bootstrap_marker_path": str(MARKER_PATH),
        "bootstrap_marker_mismatches": POST_RESTART_MARKER_MISMATCHES,
        "dependency_restart_completed": DEPENDENCY_RESTART_COMPLETED,
        "numpy_abi_preflight_passed": NUMPY_ABI_PREFLIGHT_PASSED,
        "python_version": abi_report["python_version"],
        "numpy_version": abi_report["numpy_version"],
        "numpy_path": abi_report["numpy_path"],
        "numpy_mtrand_path": abi_report["numpy_mtrand_path"],
        "torch_version": abi_report["torch_version"],
        "torch_path": abi_report["torch_path"],
        "transformers_version": transformers_module.__version__,
        "tokenizers_version": package_version("tokenizers"),
        "py_vncorenlp_version": package_version("py_vncorenlp"),
        "s1_dependency_closure_verified": S1_DEPENDENCY_CLOSURE_VERIFIED,
        "s1_import_versions": abi_report["s1_import_versions"],
        "s1_import_failures": abi_report["s1_import_failures"],
        # Complete pip check diagnostics, plus the blocking/non-blocking split.
        **DEPENDENCY_HEALTH.as_dict(),
    },
    "repository": {
        "repo_url": REPO_URL,
        "repo_ref": REPO_REF,
        "resolved_commit": RESOLVED_COMMIT,
    },
    "runtime": runtime_report,
    "corpus": corpus_report,
    "model": {
        **model_registry_report,
        "resolved_model_revision": resolved_model_revision,
        "actual_base_parameters": base_parameter_count_actual,
        "trainable_parameter_count": trainable_parameter_count,
    },
    "limits": {
        "max_train_examples": limits.max_train_examples,
        "max_validation_examples": limits.max_validation_examples,
        "max_train_batches": limits.max_train_batches,
        "max_validation_batches": limits.max_validation_batches,
        "max_optimizer_steps": limits.max_optimizer_steps,
        "batch_size": limits.batch_size,
        "max_sequence_length": limits.max_sequence_length,
        "learning_rate": limits.learning_rate,
    },
    "batch_shapes": batch_shapes,
    "tokenizer": {
        "tokenizer_class": tokenizer_report["tokenizer_class"],
        "tokenizer_is_fast": tokenizer_report["tokenizer_is_fast"],
        "tokenizer_revision": MODEL_REVISION,
        "vocab_size": tokenizer_report["vocab_size"],
    },
    "alignment": {
        "alignment_backend": ALIGNMENT_BACKEND,
        "alignment_backend_version": ALIGNMENT_BACKEND,
        "subtoken_supervision_policy": SUBTOKEN_SUPERVISION_POLICY,
        "aligned_example_count": alignment_preflight["aligned_example_count"],
        "unalignable_example_count": alignment_preflight["unalignable_example_count"],
        "truncated_example_count": alignment_preflight["truncated_example_count"],
        "truncated_entity_count": alignment_preflight["truncated_entity_count"],
        "fully_dropped_entity_count": alignment_preflight["fully_dropped_entity_count"],
        "partially_truncated_entity_count": alignment_preflight["partially_truncated_entity_count"],
        "partial_truncation_policy": PARTIAL_TRUNCATION_POLICY,
        "boundary_merge_policy": BOUNDARY_MERGE_POLICY,
        "alignment_scope": alignment_preflight["alignment_scope"],
        "examples_considered": alignment_preflight["examples_considered"],
        "boundary_merge_masked_word_count": alignment_preflight["boundary_merge_masked_word_count"],
        "boundary_merge_affected_entity_count": alignment_preflight["boundary_merge_affected_entity_count"],
        "pre_segmented_example_count": alignment_preflight["pre_segmented_example_count"],
        "segmenter_example_count": alignment_preflight["segmenter_example_count"],
        "segmentation_policy": segmenter_report["segmentation_policy"],
        "counters_reconciled": alignment_preflight["counters_reconciled"],
        "governed_exclusion_count": alignment_preflight["governed_exclusion_count"],
        "governed_exclusions": alignment_preflight["governed_exclusions"],
        "unalignable_examples": alignment_preflight["unalignable_examples"],
        "reason_code_counts": alignment_preflight["reason_code_counts"],
        **tokenizer_equivalence,
    },
    "word_segmentation": {
        "segmenter_mode": segmenter_report["segmenter_mode"],
        "degraded_fallback": segmenter_report["degraded_fallback"],
        "word_segmenter": segmenter_report["word_segmenter"],
        "word_segmenter_version": segmenter_report["word_segmenter_version"],
        "word_segmenter_resource_hashes": segmenter_report["word_segmenter_resource_hashes"],
        "acquisition_source": segmenter_report["acquisition_source"],
    },
    "loss_values": loss_values,
    "validation_metrics": validation_metrics,
    "loss_masks": {"vietmed_train_only": mask_report},
    "artifacts": {
        **smoke_artifact_paths.as_dict(),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": checkpoint_sha256,
        "training_manifest_path": str(TRAINING_MANIFEST_PATH),
    },
}
TRAINING_MANIFEST_PATH.write_text(json.dumps(training_manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({
    "status": "SMOKE_ONLY",
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": checkpoint_sha256,
    "training_manifest_path": str(TRAINING_MANIFEST_PATH),
    "checkpoint_reload_succeeded": True,
}, indent=2, sort_keys=True))


## 21. Return Artifacts

Return `OUTPUT_DIR` from Drive for review. It must contain `checkpoint/s1_mention_smoke_model.pt` and `training_manifest.json`. Do not return base-model cache files, run organizer inference, run packaging, or claim full S1 training.
